From satellite flood maps (all sources) extract for each year, all affected regions
as a list of disjoint rectangles. Overlapping flood maps within one year are merged
into one rectangular region covering both.

In addition, determine rectangular affected regions for the full period.

In [ ]:
%matplotlib widget

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from matplotlib.backends.backend_agg import FigureCanvasAgg
import matplotlib.collections as mcollections
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

import gcvalid.util as util
import gcvalid.util.constants as u_const

In [ ]:
df = pd.concat([
    pd.read_hdf(u_const.FLOODMAPS_DIR / source / "meta.hdf5")
    for source in ["gfd", "dfo", "rapid"]
])
df["date"] = pd.to_datetime(df["date"])

year_dfs = {
    "Total" if year is None else str(year): df if year is None else df[df["date"].dt.year == year]
    for year in [None] + list(np.unique(df["date"].dt.year.values))
}
comps = {
    year: util.rectangular_components([
        tuple(row)
        for row in df_yr[['xmin', 'ymin', 'xmax', 'ymax']].values
    ])
    for year, df_yr in year_dfs.items()
}

In [ ]:
rectangles = comps["Total"]
fig = plt.figure(figsize=(12, 4))
proj = ccrs.PlateCarree()
ax = fig.add_subplot(111, projection=proj)
ax.add_feature(cfeature.OCEAN.with_scale('50m'), linewidth=0.1)
pareas = mcollections.PatchCollection(
    [mpatches.Rectangle((r[0], r[1]), r[2] - r[0], r[3] - r[1])
     for r in rectangles],
    facecolor=(1, 0, 0, 0.3), edgecolor='r',
    linewidths=np.full((len(rectangles),), 0.5),
    transform=proj)
ax.add_collection(pareas)
ax.set_extent((-150, 180, -60, 70), crs=proj)
fig.tight_layout()

In [ ]:
for year, comps_yr in comps.items():
    print(year)
    for rect in comps_yr:
        print("(" + ", ".join([f"{r:.3f}" for r in rect]) + ")")
    print("")